# Assignment 2B — RAG Pipeline on the HR-policy corpus (Group 4)

A small, complete Retrieval-Augmented Generation system built on the Assignment 1 (CorpPolicyLM) domain corpus.

| Part | What is done | Marks |
|---|---|---|
| A | Dense search (FAISS) and choice of chunk size | 5 |
| B | Dense vs keyword (BM25) vs hybrid (RRF) search | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answer generation and an LLM judge | 7 |

**Reused from Assignment 1:**
* **Documents:** the 33 cleaned `.txt` policy documents (`data/domain_corpus/`, from 34 PDFs, 902 pages).
* **Queries:** 10 queries with expected answers from the Assignment 1 evaluation set (`data/a1_eval/instruction_eval.jsonl`).
* **Generator:** the fine-tuned Assignment 1 model (TinyLlama-1.1B CPT checkpoint + QLoRA Adapter B) is tried first in D1.

**Generator and judge used after the switch:** `Qwen2.5-3B-Instruct` generates and `Llama-3.2-3B-Instruct` judges, both pre-quantised to 4-bit. These are the 3B members of the two model families the brief names. The 7–8B versions (about 5.5 GB each) do not fit the lab server's 4.9 GB home disk, and the lab quota did not allow a server with a bigger volume. The code runs the 7–8B models unchanged when `GENERATOR` / `JUDGE` are set.

**Code:** the pipeline lives in `src/rag/`; this notebook calls it and shows every result:
* `data.py` — chunking and the hit test;
* `retrieval.py` — dense, BM25 and RRF search;
* `rerank.py` — cross-encoder reranking;
* `generation.py` — prompts, generator and judge.

In [1]:
import os, sys, json, time, inspect, importlib.util
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
missing = [m for m in ("faiss", "rank_bm25", "sentence_transformers", "peft", "bitsandbytes") if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"Missing packages {missing}: in a terminal run `pip install -r requirements.txt`, then restart the kernel.")
import numpy as np, pandas as pd, torch, faiss, transformers, sentence_transformers
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", None)
from src.rag import config
from src.rag.data import load_corpus, load_queries, chunk_fixed, is_hit, answer_chunk_ids, norm
from src.rag import retrieval, rerank as rr, generation as gen
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sentence-transformers",
      sentence_transformers.__version__, "| faiss", faiss.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
print(f"Hugging Face cache: {gen.hf_cache_dir()} ({gen.free_gb():.1f} GiB free)")

torch 2.5.1+cu124 | transformers 5.18.0 | sentence-transformers 6.1.0 | faiss 1.9.0
GPU: NVIDIA A100-SXM4-80GB
Hugging Face cache: /home/jovyan/.cache/huggingface/hub (0.6 GiB free)


## Before you start — queries, answer spans and the embedding model's limit
The 10 queries come from the Assignment 1 evaluation set (12 held-out pairs from 6 documents). Two were left out:
* #1 asks about the same Niramai passage as #0 with a near-identical question.
* #3 is an abstract re-reading of the same Astral sentence as #2.

For each query, the **answer span** is the **shortest contiguous source passage that supports the complete expected answer**.
* For comparison questions (Q3, Q6, Q7, Q10), the span covers **both** sides being compared. Q6, for example, runs from the individual employees' responsibilities through the managers' responsibilities.
* For Q2, Q4 and Q9 it also includes the condition or channel that the expected answer names.

Every span is cut from its source document (`tools/make_queries.py`), so it occurs verbatim. A retrieved chunk is a **hit** only if it contains the **whole** span; the comparison is case-insensitive and ignores line breaks. A chunk that holds only one side of a comparison is therefore not a hit.

*Revision note:* a first version used shorter spans, some covering only one side of a comparison. The stricter spans lower every hit rate below, and longer spans are more often split across a chunk boundary.

A separate 11th query, whose answer is not in the corpus, is used only in D1.

In [2]:
docs = load_corpus()
queries, ooc = load_queries()
print(f"{len(docs)} documents, {sum(len(t.split()) for t in docs.values()):,} words")
display(pd.DataFrame(queries)[["qid", "query", "answer_span", "span_words", "source", "expected_answer"]])
print("Out-of-corpus query (D1.3):", ooc["query"])

33 documents, 241,533 words


,qid,query,answer_span,span_words,source,expected_answer
0,Q1,Who is covered by Niramai's Code of Conduct?,is applicable to all the employees appointed on the payroll of Niramai Health Analytix Private Limited,16,niramai_code_of_conduct.txt,"The code applies to employees appointed on Niramai Health Analytix's payroll. It guides their behavior and attitudes during employment and provides ethical guidance for carrying out official responsibilities, rather than describing a separate policy for outside stakeholders."
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"mandates that every Listed Company to establish a mechanism for employees/directors to report concerns about unethical behavior, actual or suspected fraud or violation of the company’s code of conduct or ethics policy. ❖ Such vigil mechanism not only helps to detect fraud in organizations, but is also used as a corporate governance tool, which prevents and deters fraudulent activity",59,astral_whistle_blower_policy.txt,"Astral describes the mechanism as a means for directors and employees to raise concerns about unethical conduct, suspected or actual fraud, and code violations. It also supports corporate governance by helping detect and deter fraudulent activity."
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,(a) provision of a fixed monthly disbursement (“Laptop Charges”) for use of personal laptops (Bring Your Own Device / BYOD model); and (b) provision of organization -owned laptops for specific roles requiring advanced technical capabilities,35,mygov_laptop_policy.txt,"One arrangement provides a fixed monthly payment for employees using their own laptops under the BYOD model. The other supplies an organization-owned laptop for roles needing advanced technical capabilities, with applicable provisions depending on the device category allocated."
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,"based on their role, job requirements, and technical needs. Eligibility will be determined by the department head or immediate supervisor",20,mygov_laptop_policy.txt,"Eligibility depends on the employee's role, job requirements and technical needs, and is decided by the department head or immediate supervisor. The policy covers both payments for personal-laptop use and organization-owned devices, with provisions depending on the assigned category."
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"In case of a verbal complaint, the complaint will be converted to a written complaint by the receiver of the complaint and consent of the complainant will be obtained",29,bajaj_broking_posh_policy.txt,"When formal intervention is warranted, the complaint must be put in writing. If the complaint was initially verbal, the person receiving it writes it down and obtains the complainant's consent before it enters the formal redressal process."
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,"Responsibilities of Individual: It is the responsibility of all employees to respect the rights of all employees and to never encourage harassment against employees. It can be done by: R efusing to participate in any activity which constitutes harassment against any person ; Supporting the person to reject unwelcome behavior ; Acting as a witness if any person, being harassed, decides to lodge a complaint with the internal committee iv. I f you see something, say something v. If any employee notices inappropriate or unwelcome behavior in the work environment, they are encouraged to advise the person to modify or stop the same. Often, some behaviors are not intentional. While this does not make it acceptable, it does give the pe rson behaving inappropriately, the opportunity to modify or stop their offensive behavior. vi. Report t o D AC vii. If an employee notices inappropriate or unwelcome behavior in the work envir

Out-of-corpus query (D1.3): Does MyGov allow employees to bring their pet dogs to the office, and what rules apply?


In [3]:
embedder = retrieval.load_embedder()
emb_tok = embedder.tokenizer
limit = embedder.max_seq_length
doc_tokens = {name: len(emb_tok(text, add_special_tokens=False, verbose=False)["input_ids"]) for name, text in docs.items()}
print(f"Embedding model: {config.EMBED_MODEL} | max input length: {limit} tokens "
      f"(incl. [CLS]/[SEP], so up to {limit - 2} content tokens per chunk)")
print(f"Corpus size with this tokenizer: {sum(doc_tokens.values()):,} tokens "
      f"(largest document {max(doc_tokens.values()):,}, smallest {min(doc_tokens.values()):,})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: BAAI/bge-small-en-v1.5 | max input length: 512 tokens (incl. [CLS]/[SEP], so up to 510 content tokens per chunk)
Corpus size with this tokenizer: 316,899 tokens (largest document 76,882, smallest 480)


**Why not `all-MiniLM-L6-v2`?** Its limit is 256 tokens, so 400- and 500-token chunks would be silently cut to their first 256 tokens. We therefore use **`BAAI/bge-small-en-v1.5`** (384-dim, 512-token limit, the same size class as MiniLM), so every chunk size is embedded in full. Token counts are always taken with this model's own tokenizer. Following the model card, queries get the prefix *"Represent this sentence for searching relevant passages: "*; chunks are embedded as they are.

# Part A — Dense search and chunk size (5 marks)
## A1. Build dense search (2 marks)
### A1.1 One reusable function (1 mark)
`DenseIndex` embeds a set of chunks, builds a FAISS `IndexFlatIP` and returns the top-k chunks for a query.

In [4]:
print(inspect.getsource(retrieval.DenseIndex))

class DenseIndex:
    """Embeds chunks, stores them in a FAISS IndexFlatIP and returns the top-k for a query.

    Embeddings are L2-normalised, so inner product = cosine similarity.
    """

    def __init__(self, chunks, embedder, batch_size=config.EMBED_BATCH,
                 query_instruction=config.QUERY_INSTRUCTION):
        self.chunks, self.embedder, self.query_instruction = chunks, embedder, query_instruction
        self.batch_size = batch_size
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        vectors = embedder.encode([c.text for c in chunks], batch_size=batch_size,
                                  normalize_embeddings=True, convert_to_numpy=True,
                                  show_progress_bar=False).astype("float32")
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        self.build_seconds = time.perf_counter() - t0
        self.dim = vectors.shape[1]

    def 

## A2. Choose a chunk size (3 marks)
### A2.1 Fixed-size chunking at 100, 400 and 500 tokens, no overlap (2 marks)
The brief's text asks for 100 and 400 tokens, while its table lists 100 and 500, so all three sizes are run. Chunks are cut every *n* tokens of the embedding tokenizer, never across documents, and their text is taken from the original document, so a span can be matched exactly.

**Columns:**
* *Total chunks* — the absolute count (the table's "(1M)" header appears to be a typo).
* *Queries answerable* — queries whose span lies inside a single chunk; a span that straddles a chunk boundary cannot be a hit at that size.
* *Prompt tokens* — the exact length of the fully formatted chat prompt (system message, rules, the three numbered chunks and the question), counted **with each generator's own tokenizer**, plus 200 tokens reserved for the answer. It is shown as the maximum over the 10 queries, together with how many queries exceed the model's context limit:
  * Assignment 1 model (TinyLlama): 2,048 tokens;
  * Qwen2.5: 32,768 tokens.
  These rows use the dense top 3 of each size. D1 repeats the check for the exact prompts that were sent.

In [5]:
from transformers import AutoTokenizer
a1_tok = gen.a1_tokenizer()                              # Assignment 1 tokenizer + its Zephyr chat template
g_tok_count = AutoTokenizer.from_pretrained(config.GENERATOR)   # generator tokenizer only (a few MB)
G_CONTEXT = 32768                                        # Qwen2.5-3B/7B-Instruct context length
def budget(tok, size_chunks, hits):   # per-query prompt tokens (+ reserved answer tokens), top-3 chunks
    return [gen.chat_prompt_tokens(tok, gen.GEN_SYSTEM, gen.gen_prompt(q["query"], [size_chunks[c] for c, _ in h[:3]]))
            + config.GEN_MAX_NEW_TOKENS for q, h in zip(queries, hits)]
size_rows, size_runs = [], {}
for size in config.CHUNK_SIZES:
    assert size + 2 <= limit, f"{size}-token chunks exceed the embedding limit"
    chunks = chunk_fixed(docs, emb_tok, size)
    index = retrieval.DenseIndex(chunks, embedder)
    results, _ = retrieval.timed_search(index, queries, k=3)
    size_runs[size] = (chunks, index, results)
    size_rows.append({"Chunk size (tokens)": size, "Total chunks": len(chunks),
                      "Hit rate@3 (%)": retrieval.hit_at(results, queries, chunks, 3),
                      "Tokens sent per query (top 3)": round(np.mean([sum(chunks[c].n_tokens for c, _ in h[:3]) for h in results]), 1),
                      "Queries answerable (span inside one chunk)": sum(bool(answer_chunk_ids(chunks, q["answer_span"])) for q in queries),
                      "A1 prompt tokens, max (+200 answer)": max(a1_b := budget(a1_tok, chunks, results)),
                      "Queries over A1 limit (2,048)": sum(t > config.A1_CONTEXT for t in a1_b),
                      "Qwen prompt tokens, max (+200 answer)": max(g_b := budget(g_tok_count, chunks, results)),
                      "Queries over Qwen limit (32,768)": sum(t > G_CONTEXT for t in g_b),
                      "Index build time (s)": round(index.build_seconds, 2)})
size_table = pd.DataFrame(size_rows)
size_table.to_csv(config.RESULTS / "a2_chunk_sizes.csv", index=False)
display(size_table)
for size, (chunks, _, results) in size_runs.items():
    print(f"{size:>3} tokens - rank of first hit per query:",
          [retrieval.first_hit_rank(h, q, chunks) for h, q in zip(results, queries)])

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

,Chunk size (tokens),Total chunks,Hit rate@3 (%),Tokens sent per query (top 3),Queries answerable (span inside one chunk),"A1 prompt tokens, max (+200 answer)","Queries over A1 limit (2,048)","Qwen prompt tokens, max (+200 answer)","Queries over Qwen limit (32,768)",Index build time (s)
0,100,3183,20.0,300.0,3,973,0,824,0,2.41
1,400,810,70.0,1172.8,9,2081,2,1764,0,1.64
2,500,649,70.0,1457.2,9,2464,8,2058,0,1.61


100 tokens - rank of first hit per query: [1, None, 3, None, None, None, None, None, None, None]
400 tokens - rank of first hit per query: [1, 1, 2, 1, None, None, None, 1, 2, 1]
500 tokens - rank of first hit per query: [1, 1, 2, 1, None, None, 1, None, 2, 1]


### Choosing the size (rule applied in code; `CHUNK_SIZE=<n>` overrides it)
1. Every prompt must fit the context of the generator that answers in D1 (Qwen2.5, 32,768 tokens), measured per query with its own tokenizer.
2. Among those sizes, the highest hit rate@3.
3. On a tie, fewer tokens sent per query.

The Assignment 1 model's 2,048-token limit is reported in the table rather than used as a filter. That model is only the first trial in D1; how its limit interacts with the chosen size is discussed in A2.2 and D1.

In [6]:
def pick_size(table):
    ok = table[table["Queries over Qwen limit (32,768)"] == 0]
    best = ok.sort_values(["Hit rate@3 (%)", "Tokens sent per query (top 3)"], ascending=[False, True]).iloc[0]
    return int(best["Chunk size (tokens)"])
CHUNK_SIZE = config.CHUNK_SIZE or pick_size(size_table)
chunks, dense, _ = size_runs[CHUNK_SIZE]
print(f"Chosen chunk size: {CHUNK_SIZE} tokens ({len(chunks)} chunks)")

Chosen chunk size: 400 tokens (810 chunks)


### A1.2 Dense-search report for the final chunk set (1 mark)
Build time = embedding all chunks plus adding them to the FAISS index.

In [7]:
a1_report = dense.report()
json.dump(a1_report, open(config.RESULTS / "a1_dense_report.json", "w"), indent=2)
display(pd.Series(a1_report).to_frame("value"))

,value
embedding model,BAAI/bge-small-en-v1.5
vector size,384
similarity,"cosine (inner product of L2-normalised vectors, FAISS IndexFlatIP)"
max input tokens,512
batch size,64
chunks indexed,810
device,cuda
index build time (s),1.64


### A2.2 Chosen chunk size and why (1 mark)
**Chosen: 400 tokens.** 400 and 500 tie on hit rate@3 (70%), and each keeps the complete answer span inside one chunk for 9 of the 10 queries. 400 sends fewer tokens to the generator (1,173 instead of 1,457 per query for the top 3). Its longest prompt, plus 200 answer tokens, is 1,764 Qwen tokens, far below Qwen's 32,768-token limit.
* **100 tokens** is too small. Only 3 of the 10 answer spans fit inside a single chunk, so hit rate@3 drops to 20%.
* **500 tokens** gains nothing over 400 and makes the prompts longer. Measured with the Assignment 1 tokenizer, 8 of the 10 prompts would exceed the A1 model's 2,048-token context once 200 answer tokens are reserved. With 400 tokens, 2 of 10 would still exceed it (max 2,081). This is one reason the A1 model is not the final generator (see D1).

# Part B — Keyword and hybrid search (4 marks)
All three methods use the chunks chosen in Part A and return the top 5 for each of the same 10 queries.
* **Latency:** the time to return the top 5 for one query, averaged over the 10 queries, after one untimed warm-up query.
* **Hybrid:** fuses the **top 20** of each method with RRF(d) = 1/(60 + rank in dense list) + 1/(60 + rank in BM25 list). Ranks start at 1, and a chunk missing from one list gets nothing from that list.

In [8]:
print(inspect.getsource(retrieval.rrf))
bm25 = retrieval.BM25Index(chunks)
hybrid = retrieval.HybridIndex(dense, bm25)
methods = {"Dense": dense, "BM25": bm25, "Hybrid (RRF)": hybrid}
runs, rows = {}, []
for name, index in methods.items():
    results, times = retrieval.timed_search(index, queries, k=5)
    runs[name] = results
    rows.append({"Method": name, "Average query latency (ms)": round(float(np.mean(times)), 2),
                 "Hit rate@1 (%)": retrieval.hit_at(results, queries, chunks, 1),
                 "Hit rate@3 (%)": retrieval.hit_at(results, queries, chunks, 3),
                 "Hit rate@5 (%)": retrieval.hit_at(results, queries, chunks, 5)})
search_table = pd.DataFrame(rows)
search_table.to_csv(config.RESULTS / "b_search_comparison.csv", index=False)
display(search_table)
ranks = pd.DataFrame({"qid": [q["qid"] for q in queries], "query": [q["query"] for q in queries],
                      **{f"{m}: rank of first hit": [retrieval.first_hit_rank(h, q, chunks) for h, q in zip(runs[m], queries)] for m in runs}})
display(ranks)

def rrf(dense_hits, bm25_hits, k=config.RRF_K, top=config.TOP_K):
    """Reciprocal Rank Fusion: score(d) = 1/(k + rank_dense) + 1/(k + rank_bm25), ranks start at 1;
    a chunk missing from one list gets nothing from that list."""
    scores = {}
    for hits in (dense_hits, bm25_hits):
        for rank, (cid, _) in enumerate(hits, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    fused = sorted(scores.items(), key=lambda kv: -kv[1])
    return fused[:top]



,Method,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%),Hit rate@5 (%)
0,Dense,8.35,50.0,70.0,80.0
1,BM25,1.96,70.0,90.0,90.0
2,Hybrid (RRF),9.93,60.0,80.0,80.0


,qid,query,Dense: rank of first hit,BM25: rank of first hit,Hybrid (RRF): rank of first hit
0,Q1,Who is covered by Niramai's Code of Conduct?,1.0,1.0,1.0
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,1.0,1.0,1.0
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,2.0,1.0,1.0
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,1.0,1.0,1.0
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,NaN,2.0,NaN
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,NaN,NaN,NaN
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,4.0,1.0,2.0
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,1.0,1.0,1.0
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,2.0,2.0,2.0
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,1.0,1.0,1.0


### Best method (rule applied in code; `BEST_METHOD=<name>` overrides it)
The best method has the highest hit rate@3; ties are broken by hit rate@1, then by lower latency.

In [9]:
best_row = search_table.sort_values(["Hit rate@3 (%)", "Hit rate@1 (%)", "Average query latency (ms)"],
                                    ascending=[False, False, True]).iloc[0]
BEST = os.environ.get("BEST_METHOD") or best_row["Method"]
print("Best method for Part C:", BEST)

Best method for Part C: BM25


### B4. Which method works best and why, with one example query (1 mark)
**BM25 works best:** hit rate@1 70%, @3 90%, @5 90%, against 50 / 70 / 80 for dense search and 60 / 80 / 80 for hybrid. It is also the fastest (1.96 ms per query).

The queries name specific organisations and use the policies' own terms, such as "verbal complaint", "JioStar" and "Aurobindo". Exact keyword matching rewards these terms. The small general-purpose embedding model instead finds chunks that are similar in topic, and many policies in this corpus share topics (for example, five different harassment policies).

**Example, Q5** ("How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?"): BM25 ranks the span chunk 2nd, because it contains the exact words "verbal complaint", "converted" and "written complaint". Dense search does not have it in its top 5; it returns other harassment-procedure chunks. Hybrid search does not have it in its top 5 either: the dense ranking pushes it below rank 5 in the fusion. Hybrid helps only when both methods rank the chunk fairly high.

# Part C — Reranking (4 marks)
### C1. Rerank (1 mark)
The best Part B method returns the top 10 for each query. A cross-encoder, `cross-encoder/ms-marco-MiniLM-L-6-v2`, re-scores each (query, chunk) pair and the top 3 are kept. Reranking time per query covers scoring the 10 pairs and sorting them, averaged after one warm-up query.

In [10]:
reranker = rr.load_reranker()
before10, _ = retrieval.timed_search(methods[BEST], queries, k=10)
after3, rerank_ms = rr.timed_rerank(reranker, queries, before10, chunks, keep=3)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### C2. Top 3 before and after reranking for each query (1 mark)
`[chunk id] source`; ✓ marks a chunk that contains the answer span.

In [11]:
def show(hits, q):
    return " | ".join(f"[{c}] {chunks[c].source}{' ✓' if is_hit(chunks[c].text, q['answer_span']) else ''}" for c, _ in hits[:3])
compare = pd.DataFrame([{"qid": q["qid"], "query": q["query"], f"top 3 before ({BEST})": show(b, q),
                         "top 3 after reranking": show(a, q), "first result changed": b[0][0] != a[0][0]}
                        for q, b, a in zip(queries, before10, after3)])
compare.to_csv(config.RESULTS / "c_rerank_comparison.csv", index=False)
display(compare)

,qid,query,top 3 before (BM25),top 3 after reranking,first result changed
0,Q1,Who is covered by Niramai's Code of Conduct?,[793] niramai_code_of_conduct.txt ✓ | [48] aubank_whistle_blower_policy.txt | [801] niramai_code_of_conduct.txt,[793] niramai_code_of_conduct.txt ✓ | [58] bajaj_broking_posh_policy.txt | [801] niramai_code_of_conduct.txt,False
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,[40] astral_whistle_blower_policy.txt ✓ | [78] beml_whistle_blower_policy.txt | [256] growxcd_vigil_mechanism_policy.txt,[40] astral_whistle_blower_policy.txt ✓ | [654] khannapaper_vigil_mechanism_policy.txt | [78] beml_whistle_blower_policy.txt,False
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,[783] mygov_laptop_policy.txt ✓ | [785] mygov_laptop_policy.txt | [780] mygov_internship_program.txt,[792] mygov_talent_development_policy.txt | [783] mygov_laptop_policy.txt ✓ | [790] mygov_talent_development_policy.txt,True
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,[783] mygov_laptop_policy.txt ✓ | [784] mygov_laptop_policy.txt | [786] mygov_laptop_policy.txt,[783] mygov_laptop_policy.txt ✓ | [785] mygov_laptop_policy.txt | [784] mygov_laptop_policy.txt,False
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,[57] bajaj_broking_posh_policy.txt | [63] bajaj_broking_posh_policy.txt ✓ | [56] bajaj_broking_posh_policy.txt,[57] bajaj_broking_posh_policy.txt | [63] bajaj_broking_posh_policy.txt ✓ | [56] bajaj_broking_posh_policy.txt,False
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,[57] bajaj_broking_posh_policy.txt | [56] bajaj_broking_posh_policy.txt | [110] ccs_conduct_rules_1964.txt,[57] bajaj_broking_posh_policy.txt | [56] bajaj_broking_posh_policy.txt | [58] bajaj_broking_posh_policy.txt,False
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,[652] jiostar_posh_policy.txt ✓ | [59] bajaj_broking_posh_policy.txt | [58] bajaj_broking_posh_policy.txt,[646] jiostar_posh_policy.txt | [59] bajaj_broking_posh_policy.txt | [645] jiostar_posh_policy.txt,True
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,[652] jiostar_posh_policy.txt ✓ | [202] dav_university_hr_employee_manual.txt | [6] 3m_posh_policy.txt,[652] jiostar_posh_policy.txt ✓ | [6] 3m_posh_policy.txt | [646] jiostar_posh_policy.txt,False
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,[51] aurobindo_human_rights_policy.txt | [54] aurobindo_human_rights_policy.txt ✓ | [53] aurobindo_human_rights_policy.txt,[54] aurobindo_human_rights_policy.txt ✓ | [52] aurobindo_human_rights_policy.txt | [53] aurobindo_human_rights_policy.txt,True
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,[54] aurobindo_human_rights_policy.txt ✓ | [71] bajaj_broking_posh_policy.txt | [337] iim_raipur_hr_policy_service_rules.txt,[54] aurobindo_human_rights_policy.txt ✓ | [53] aurobindo_human_rights_policy.txt | [51] aurobindo_human_rights_policy.txt,False


### C3. Reranking measures (1 mark)
*Rank change rate* is the share of the 10 queries whose first result changes after reranking.

In [12]:
rerank_table = pd.DataFrame({"Measure": ["Reranking time per query (ms)", "Rank change rate (%)",
                                         f"Hit rate@1 before reranking (from Part B, {BEST})", "Hit rate@1 after reranking",
                                         f"Hit rate@3 before reranking ({BEST})", "Hit rate@3 after reranking"],
                             "Value": [round(float(np.mean(rerank_ms)), 1), 100 * compare["first result changed"].mean(),
                                       retrieval.hit_at(before10, queries, chunks, 1), retrieval.hit_at(after3, queries, chunks, 1),
                                       retrieval.hit_at(before10, queries, chunks, 3), retrieval.hit_at(after3, queries, chunks, 3)]})
rerank_table.to_csv(config.RESULTS / "c_rerank_measures.csv", index=False)
display(rerank_table)

,Measure,Value
0,Reranking time per query (ms),21.3
1,Rank change rate (%),30.0
2,"Hit rate@1 before reranking (from Part B, BM25)",70.0
3,Hit rate@1 after reranking,60.0
4,Hit rate@3 before reranking (BM25),90.0
5,Hit rate@3 after reranking,80.0


### C4. Did reranking help? (1 mark)
**No, on these 10 queries reranking made the results worse.** Hit rate@1 fell from 70% to 60% and hit rate@3 from 90% to 80%. The first result changed for 3 of 10 queries (30%), and reranking cost 21.3 ms per query.
* **Q9 improved:** the span chunk moved from rank 2 to rank 1.
* **Q3 got worse at rank 1:** a talent-development chunk about course-fee support ("financial support", "75%") took first place, and the laptop-policy span chunk moved to rank 2. It is still in the top 3.
* **Q7 was lost:** BM25 ranked the span chunk 652 first, but the cross-encoder put two other JioStar chunks (646, 645) and a Bajaj chunk above it, so it fell out of the top 3.

The reranker (ms-marco-MiniLM-L-6-v2) was trained on web search questions. It rewards chunks that sound like an answer to the question in general, and it does not know that the organisation name matters. BM25's exact term matching was already the stronger signal for this corpus.

# Part D — Generate and check answers (7 marks)
## D1. Generate grounded answers (3 marks)
### D1.1 The prompt (1 mark)
Each query gets the **top 3 chunks after reranking**, numbered [1]–[3]. Decoding is greedy (temperature 0).

In [13]:
print("SYSTEM:", gen.GEN_SYSTEM, "\n")
print(gen.GEN_TEMPLATE)

SYSTEM: You are an HR-policy assistant. You answer questions strictly from the document chunks you are given and never use outside knowledge. 

Answer the question using ONLY the numbered chunks below.

Rules:
1. Use only facts stated in the chunks. Do not add facts from outside knowledge.
2. The chunks are labelled [1], [2] and [3]. After each sentence, cite the label(s) of the chunk(s) you used, for example [2]. Cite ONLY these labels: numbers that appear inside the chunk text, such as 5.2 or 4.1.1, are policy section numbers, not chunk labels, and must never be cited.
3. If the chunks do not contain the answer, reply exactly: Not found in the documents.
4. Answer in 1 to 3 sentences.

Chunks:
{chunks}

Question: {question}
Answer:


In [14]:
# retrieval for the out-of-corpus query uses the same pipeline (best method top 10 -> rerank -> top 3)
ooc_q = {"qid": ooc["qid"], "query": ooc["query"], "expected_answer": ooc["expected_answer"], "answer_span": ""}
ooc_after3 = rr.rerank(reranker, ooc_q["query"], methods[BEST].search(ooc_q["query"], 10), chunks, keep=3)
all_q, all_ctx = queries + [ooc_q], after3 + [ooc_after3]
contexts = [[chunks[c] for c, _ in hits] for hits in all_ctx]
prompts = [gen.gen_prompt(q["query"], ctx) for q, ctx in zip(all_q, contexts)]
del embedder, reranker; gen.free()
print(f"Prompt length for Q1 with {CHUNK_SIZE}-token chunks: {len(prompts[0].split())} words")

Prompt length for Q1 with 400-token chunks: 843 words


### First try: the Assignment 1 model (TinyLlama-1.1B CPT + Adapter B)
The brief allows switching to an instruct model if the Assignment 1 model ignores the chunks or never says "Not found". The same prompts are run through it first, and three behaviours are checked automatically: whether it cites a chunk, whether it says "Not found", and whether the answer stops cleanly.

In [15]:
trial = json.loads((config.A1_TRIAL_DIR / "a1_trial_contexts.json").read_text())
if (config.A1_BASE / "config.json").exists() and (config.A1_ADAPTER / "adapter_config.json").exists():
    a1_model, a1m_tok = gen.load_a1_model()
    a1_prompts, source_note = prompts, "generated in this run with the current prompt"
    a1_answers = pd.DataFrame([{"qid": q["qid"], "query": q["query"], "A1 model answer": gen.chat(a1_model, a1m_tok, gen.GEN_SYSTEM, p)}
                               for q, p in zip(all_q, a1_prompts)])
    del a1_model; gen.free()
else:   # the A1 model files were removed from the lab disk after its trial run: show the saved trial
    tchunks = size_runs[trial["chunk_size"]][0]
    for q in all_q:   # the saved chunk ids must point at the same documents in this run's chunking
        ids = trial["contexts"][q["qid"]]["chunk_ids"]
        assert all(c < len(tchunks) for c in ids) and [tchunks[c].source for c in ids] == trial["contexts"][q["qid"]]["sources"], \
            f"saved A1 trial contexts do not match this run's {trial['chunk_size']}-token chunks ({q['qid']})"
    a1_prompts = [gen.GEN_TEMPLATE_V1.format(chunks=gen.format_chunks([tchunks[c] for c in trial["contexts"][q["qid"]]["chunk_ids"]]),
                                             question=q["query"]) for q in all_q]
    a1_answers = pd.read_csv(config.A1_TRIAL_DIR / "d1_a1_model_answers.csv")[["qid", "query", "A1 model answer"]]
    source_note = (f"saved from its trial run ({trial['chunk_size']}-token chunks, BM25 top 10 -> reranked top 3, "
                   "prompt version 1 - before the citation-label sentence); the model files are no longer on the lab disk")
a1_answers["cites a chunk"] = [bool(gen.citation_check(a)[0]) for a in a1_answers["A1 model answer"]]
a1_answers["says Not found"] = [gen.says_not_found(a) for a in a1_answers["A1 model answer"]]
# exact prompt length with the A1 tokenizer and chat template, + 200 tokens reserved for the answer
a1_answers["A1 prompt tokens"] = [gen.chat_prompt_tokens(a1_tok, gen.GEN_SYSTEM, p) for p in a1_prompts]
a1_answers["prompt + 200 within 2,048"] = a1_answers["A1 prompt tokens"] + config.GEN_MAX_NEW_TOKENS <= config.A1_CONTEXT
a1_answers.to_csv(config.RESULTS / "d1_a1_model_answers.csv", index=False)
print("Assignment 1 model answers:", source_note)
display(a1_answers)
over = a1_answers[~a1_answers["prompt + 200 within 2,048"]]
print(f"A1 prompt tokens: max {a1_answers['A1 prompt tokens'].max()}, {len(over)}/11 prompts leave less than 200 tokens "
      f"for the answer within the 2,048-token context ({', '.join(over.qid) or 'none'})")
print(f"A1 model: cites a chunk in {a1_answers['cites a chunk'].sum()}/11 answers; "
      f"says 'Not found' for the out-of-corpus query: {bool(a1_answers['says Not found'].iloc[-1])}")

Assignment 1 model answers: saved from its trial run (500-token chunks, BM25 top 10 -> reranked top 3, prompt version 1 - before the citation-label sentence); the model files are no longer on the lab disk


,qid,query,A1 model answer,cites a chunk,says Not found,A1 prompt tokens,"prompt + 200 within 2,048"
0,Q1,Who is covered by Niramai's Code of Conduct?,"The Code of Conduct applies to all employees, including temporary, part-time, and casual staff. It also applies to contractors and consultants.",False,False,1992,False
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"The mechanism is intended tob, and the whistle-purchase|\n\nThe mechanism, the whistle-purchase and\n\nThe mechanism, and the Company policy\nThe whistle purchase\n\nThe mechanism\n\nThe Company\n\nThe purchase\n\nThe mechanism\n\n\n\n\n\n\n\n\n\n\n\n\nThe purchase\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nor and and and and and ors or or or or or or or or",False,False,2195,False
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,"The organization provides a fixed monthly payment for the use of a laptop, but the employee is expected to cover the cost of the laptop's purchase and maintenance. The policy also provides for reimbursement of the cost of the laptop's purchase and maintenance, but the employee is expected to cover the cost of the laptop's purchase and maintenance.",False,False,1992,False
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,The eligibility for laptop charges is based on the employee’s role and responsibilities. The eligibility for laptop charges is also based on the employee’s role and responsibilities. The eligibility for laptop charges is also based on the employee’s role and responsibilities.,False,False,2040,False
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"The IC will receive the complaint and refer it to the IC for the workplace, which will manage the redressal process. The IC wills the process, which is sexual harassment at work, and handle the redressal, which is the IC, which the work and manage the redressal, and the IC.\n\nThe IC, which work and handle the redness.",False,False,2167,False
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,"The company's policy is to prevent, and take disciplinary action against the perpetrator, and to provide for the aggrieved person's name, address, telephone number, and email address, and to provide the policy's provisions, and the company's policy, and disciplinary action, and providing the aggrants, and providing, and email, and email, and sms, and email, and providing, and providing, and providing, and providing, and providing, and providing ands, and providing, andsafact or providing, and providing, and providing and providing ands.\n\nand providing, providing, and providing ands\nand providing and\n\nand\nands\n\n\nand and andand ands ands ands ands ands ands ands andand ands ands ands and and ands ands ands ands ands ands ands ands and and and",False,False,2165,False
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,"The scope of the policy includes verbal conduct such as sexually suggestive or offensive remarks, jokes, inappropriate comments about a person's clothing or body, and physical conduct such as unwanted physical contact or physical assault. It also covers unwelcome physical contact such as grabbing, pinching, or pushing.",False,False,1989,False
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,"The policy covers sexual harassment, unwelcome sexually determined conduct, and retaliation against individuals who report or participate in an investigation. The policy also prohibits retaliation against witnesses and provides for disciplinary action against perpetrators.",False,False,1783,True
8,Q9,What initial action does Aurobindo describe after receiving a human-rights co

A1 prompt tokens: max 2195, 10/11 prompts leave less than 200 tokens for the answer within the 2,048-token context (Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q9, Q10, Q11)
A1 model: cites a chunk in 0/11 answers; says 'Not found' for the out-of-corpus query: False


**Problem observed with the Assignment 1 model, and the switch.** The Assignment 1 model (TinyLlama-1.1B with the CPT and Adapter B) is shown with the answers saved from its trial run. Its model files are no longer on the lab disk, so it could not be run again with the revised prompt. **That trial used 500-token chunks and prompt version 1** (before the sentence on chunk labels was added). Both are stated in the table above, and its prompts were rebuilt exactly to count tokens with its own tokenizer.

**Problems observed:**
1. **The context is too long for the model.** TinyLlama has a 2,048-token context. The prompts of Q2, Q5 and Q6 are 2,165–2,195 tokens, so they exceed the limit before any answer is written. These are exactly the answers that break down into repeated or meaningless text (Q2: "The mechanism… The purchase…"). Q4 (2,040 tokens, leaving 8 for the answer) repeats one sentence. Only Q8 leaves the 200 answer tokens. This is evidence that the overlong context contributed to the failures.
2. **It does not use the chunks even when the prompt fits.** It cites no chunk in any of the 11 answers. Q8, the one prompt that fits, gives a plausible answer, but it cites nothing and adds details ("retaliation against witnesses") that are not in the span.
3. **It invents an answer for the out-of-corpus query.** For Q11 it describes a pet-friendly office with a leash rule. No such policy exists in the corpus.

The model was trained to answer HR questions from its own memory, not to follow a context-and-citation instruction. It is also too small for that task. **Switch:** the generator is Qwen2.5-3B-Instruct (4-bit), which has a 32,768-token context and is instruction-tuned. A 7B/8B model was preferred but did not fit the lab disk (4.9 GB home volume). With 400-token chunks, the longest Qwen prompt is 1,764 tokens including 200 answer tokens.

### D1.2 Answers of the generator (1 mark) and D1.3 the out-of-corpus query (1 mark)

In [16]:
print(f"Generator: {config.GENERATOR_NAME}  [{config.GENERATOR}]  - free disk before download: {gen.free_gb():.1f} GiB")
gen.make_room(config.GENERATOR, drop=config.JUDGE)   # re-runs: the judge may still be cached
g_model, g_tok = gen.load_instruct(config.GENERATOR)
answers = [gen.chat(g_model, g_tok, gen.GEN_SYSTEM, p) for p in prompts]
del g_model; gen.free()
cit = [gen.citation_check(a, n_chunks=len(ctx)) for a, ctx in zip(answers, contexts)]
gen_table = pd.DataFrame([{"qid": q["qid"], "query": q["query"], "answer": a, "valid citations": v, "invalid citations": inv,
                           "prompt tokens (Qwen)": gen.chat_prompt_tokens(g_tok, gen.GEN_SYSTEM, p),
                           "chunks used (source)": " | ".join(f"[{i}] {c.source}" for i, c in enumerate(ctx, 1))}
                          for q, a, (v, inv), p, ctx in zip(all_q, answers, cit, prompts, contexts)])
gen_table.to_csv(config.RESULTS / "d1_generator_answers.csv", index=False)
json.dump([{"qid": q["qid"], "query": q["query"], "expected_answer": q["expected_answer"], "answer": a,
            "chunk_ids": [c for c, _ in h]} for q, a, h in zip(all_q, answers, all_ctx)],
          open(config.RESULTS / "d1_answers.json", "w"), indent=2, ensure_ascii=False)
display(gen_table.iloc[:10])
print("D1.3 - out-of-corpus query:", ooc_q["query"])
print("Answer:", answers[-1])
print("Says 'Not found in the documents':", gen.says_not_found(answers[-1]))
bad = gen_table[gen_table["invalid citations"].str.len() > 0]
uncited = gen_table[(gen_table["valid citations"].str.len() == 0) & ~gen_table.answer.map(gen.says_not_found)]
print(f"Citation check: {len(bad)} answers cite labels other than [1]-[3] ({', '.join(bad.qid) or 'none'}); "
      f"{len(uncited)} answers that are not 'Not found' cite no valid chunk ({', '.join(uncited.qid) or 'none'}). "
      "Answers are shown exactly as generated.")
print(f"Qwen prompt tokens: max {gen_table['prompt tokens (Qwen)'].max()} (+200 for the answer) of a 32,768-token context")

Generator: Qwen2.5-3B-Instruct (4-bit NF4)  [unsloth/Qwen2.5-3B-Instruct-bnb-4bit]  - free disk before download: 0.6 GiB


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,qid,query,answer,valid citations,invalid citations,prompt tokens (Qwen),chunks used (source)
0,Q1,Who is covered by Niramai's Code of Conduct?,"Niramai's Code of Conduct is applicable to all employees appointed on the payroll of Niramai Health Analytix Private Limited, including staff, full-time employees both permanent and on probation, temporary employees, trainees, management trainees, apprentices, interns, and third party employees. [1]",[1],[],1165,[1] niramai_code_of_conduct.txt | [2] bajaj_broking_posh_policy.txt | [3] niramai_code_of_conduct.txt
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,"Astral Limited established a vigil mechanism and whistle-blower policy to encourage employees and directors to report unethical behaviors, malpractices, wrongful conduct, fraud, violation of the company's code of conduct or ethics policy, and violations of law by any employee/director of ASTRAL without fear of retaliation. [1]",[1],[],1529,[1] astral_whistle_blower_policy.txt | [2] khannapaper_vigil_mechanism_policy.txt | [3] beml_whistle_blower_policy.txt
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,"The laptop support arrangement differs in that the Learning and Development Policy provides financial support up to 75% of the course fee or INR 25,000, whichever is lower, whereas the Laptop Provision/Charges Policy offers a fixed monthly disbursement of INR 5,000 for personal laptops.",[],[],1406,[1] mygov_talent_development_policy.txt | [2] mygov_laptop_policy.txt | [3] mygov_talent_development_policy.txt
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,"Eligibility for a laptop or laptop charges at MyGov is determined by the department head or immediate supervisor based on the employee's role, job requirements, and technical needs. [1]",[1],[],1406,[1] mygov_laptop_policy.txt | [2] mygov_laptop_policy.txt | [3] mygov_laptop_policy.txt
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,"A verbal sexual-harassment complaint is converted to a written complaint by the receiver of the complaint, and the consent of the complainant is obtained for this conversion. [2]",[2],[],1584,[1] bajaj_broking_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] bajaj_broking_posh_policy.txt
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,"Managers at Bajaj Broking have additional responsibilities beyond individual employees regarding the prevention of sexual harassment. Specifically, the IC (Investigation Committee) while conducting inquiries can rely on the Firm's code of conduct policy (MISCONDUCT) in recommending actions to Management. This means that managers have broader oversight and influence over the handling of complaints and investigations compared to individual employees. [3]",[3],[],1564,[1] bajaj_broking_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] bajaj_broking_posh_policy.txt
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,Not found in the documents.,[],[],1433,[1] jiostar_posh_policy.txt | [2] bajaj_broking_posh_policy.txt | [3] jiostar_posh_policy.txt
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,Not found in the documents,[],[],1425,[1] jiostar_posh_policy.txt | [2] 3m_posh_policy.txt | [3] jiostar_posh_policy.txt
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,"After receiving a human-rights complaint, Aurobindo describes that preliminary enquiry will be conducted by involving the complainant in a way that the matter will be dealt with utmost confidentiality within 3 days time. [1]",[1],[],1445,[1] aurobindo_human_rights_policy.txt | [2] aurobindo_human_rights_policy.txt | [3] aurobindo_human_rights_policy.txt
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievanc

D1.3 - out-of-corpus query: Does MyGov allow employees to bring their pet dogs to the office, and what rules apply?
Answer: Not found in the documents
Says 'Not found in the documents': True
Citation check: 0 answers cite labels other than [1]-[3] (none); 1 answers that are not 'Not found' cite no valid chunk (Q3). Answers are shown exactly as generated.
Qwen prompt tokens: max 1584 (+200 for the answer) of a 32,768-token context


## D2. Check the answers with an LLM judge (4 marks)
### D2.1 Judge prompt and results (1 mark)
**Judge model:** Llama (printed below), a different model family from the Qwen generator. Decoding is greedy, so it behaves like temperature 0 and gives the same verdict every time.

The judge is given the question, the expected answer from the Assignment 1 evaluation set, the three retrieved chunks and the generator's answer. It returns four fixed lines (`CORRECT`, `SUPPORTED`, `REASON`, `UNSUPPORTED`), which are parsed with code.

**Constrained decoding of the format.** On a first run, the 3B judge left the CORRECT and SUPPORTED fields empty in all 10 replies. The reply is therefore built line by line from the same prompt:
* **Labels:** for `CORRECT` and `SUPPORTED`, the judge chooses among the allowed labels only, taking the label whose first token it rates most probable. This is argmax, i.e. temperature 0.
* **Explanations:** `REASON` and `UNSUPPORTED` are then generated greedily, conditioned on the chosen labels.

In [17]:
print("SYSTEM:", gen.JUDGE_SYSTEM, "\n")
print(gen.JUDGE_TEMPLATE)

SYSTEM: You are a strict examiner who grades answers from an HR-policy question-answering system. 

Grade the SYSTEM ANSWER below. Check two things separately.

CHECK 1 - CORRECT: compare the system answer with the EXPECTED ANSWER.
- correct: it states the main point(s) of the expected answer and contradicts nothing in it.
- partly correct: it states some of the expected answer's points but misses an important one or adds an error, without contradicting the main point.
- wrong: it misses or contradicts the main point, answers a different question, or is "Not found in the documents" when the expected answer exists.
If the expected answer is itself "Not found in the documents", then a "Not found" reply is correct and any invented answer is wrong.
Ignore citation numbers and wording; judge the meaning.

CHECK 2 - SUPPORTED: compare every statement in the system answer with the RETRIEVED CHUNKS only.
- yes: every statement can be found in (or directly follows from) the chunks.
- no: at lea

In [18]:
gen.make_room(config.JUDGE, drop=config.GENERATOR)   # small disk: one model at a time (answers are already saved)
print(f"Judge: {config.JUDGE_NAME}  [{config.JUDGE}]")
j_model, j_tok = gen.load_instruct(config.JUDGE)
judged = [gen.judge(j_model, j_tok, gen.judge_prompt(q["query"], q["expected_answer"], ctx, a))
          for q, a, ctx in zip(queries, answers[:10], contexts[:10])]
del j_model; gen.free()
verdicts = pd.DataFrame([{"qid": q["qid"], "query": q["query"], **v} for q, (v, _) in zip(queries, judged)])
raw = [r for _, r in judged]
verdicts["raw judge output"] = raw
verdicts.to_csv(config.RESULTS / "d2_judge_verdicts.csv", index=False)
display(verdicts[["qid", "query", "correct", "supported", "reason", "unsupported quote"]])
summary = pd.DataFrame({"Check": ["Correct", "Partly correct", "Wrong", "Supported by the chunks"],
                        "Count out of 10": [int((verdicts.correct == "correct").sum()), int((verdicts.correct == "partly correct").sum()),
                                            int((verdicts.correct == "wrong").sum()), int((verdicts.supported == "yes").sum())]})
display(summary)
print("Unparsed verdicts:", int((verdicts[["correct", "supported"]] == "unparsed").any(axis=1).sum()))

Judge: Llama-3.2-3B-Instruct (4-bit NF4)  [unsloth/Llama-3.2-3B-Instruct-bnb-4bit]


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,qid,query,correct,supported,reason,unsupported quote
0,Q1,Who is covered by Niramai's Code of Conduct?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the scope of applicability, which includes third-party employees, and adds an error, stating ""including third party employees"" which is not mentioned in the retrieved chunks.","""including third party employees"""
1,Q2,Why did Astral establish a vigil mechanism and whistle-blower policy?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the objective of the policy, which is to extract maximum information on activities affecting the company, its values, philosophy, principles, and beliefs, including violations of legal or regulatory requirements.","""without fear of retaliation"""
2,Q3,How do the two laptop-support arrangements in MyGov's policy differ?,partly correct,yes,The system answer correctly identifies two key points from the expected answer but misses the main point about the BYOD model and the organization-owned laptop for advanced technical capabilities.,None
3,Q4,How is eligibility for a laptop or laptop charges determined at MyGov?,partly correct,yes,"The system answer partially states the expected answer's points but misses an important one, specifically the provisions depending on the assigned category.",None
4,Q5,How is a verbal sexual-harassment complaint converted for formal handling at Bajaj Broking?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the requirement for the complainant to be informed of their rights and the process for lodging a complaint.","""the consent of the complainant will be obtained"""
5,Q6,How do Bajaj Broking managers' harassment-prevention duties differ from individual employees' duties?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the managers' duties to ensure their teams understand zero tolerance and prevent victimization of complainants, respondents, and witnesses.","Specifically, the IC (Investigation Committee)"
6,Q7,How does JioStar's listed harassment scope distinguish verbal from visual conduct?,wrong,no,"The system answer ""Not found in the documents"" contradicts the main point of the expected answer, which states the policy covers both spoken behavior and material displayed or shown in the workplace.",None
7,Q8,Does JioStar's harassment scope include retaliation for taking part in an investigation?,partly correct,yes,"The system answer states that retaliation for taking part in an investigation is included in the scope of the policy, which is supported by the retrieved chunks.",None
8,Q9,What initial action does Aurobindo describe after receiving a human-rights complaint?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the description of the preliminary enquiry process involving Corporate HR, Unit HR, concerned Department HOD, and the detailed enquiry by having aggrieved employee, contractors, suppliers, vendors from whom the complaint is received.",None
9,Q10,How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?,partly correct,yes,"The system answer states some of the expected answer's points but misses an important one, specifically the distinction between preliminary and detailed enquiry in terms of the parties involved and the nature of the investigation.",None


,Check,Count out of 10
0,Correct,0
1,Partly correct,9
2,Wrong,1
3,Supported by the chunks,9


Unparsed verdicts: 0


### D2.2 Spot-check of the judge (1 mark)
I read the full answer, the expected answer and the three chunks for **Q4, Q5 and Q8**. Each has two verdicts (correct, supported), so six in total. **I agree with 3 of the 6.**

| Query | Judge: correct / supported | My verdict | Agree? |
|---|---|---|---|
| Q4 MyGov laptop eligibility | partly correct / yes | partly correct / yes | 2 of 2 |
| Q5 Bajaj verbal complaint | partly correct / yes | **correct** / yes | 1 of 2 |
| Q8 JioStar retaliation | partly correct / yes | **wrong** / **yes** | 0 of 2 (see below) |

* **Q4 (agree):** The answer gives who decides (department head or immediate supervisor) and the criteria (role, job requirements, technical needs), both from chunk [1]. It leaves out the second half of the expected answer, the two arrangements (BYOD allowance and organisation-owned laptop) that depend on the device category. So *partly correct* and *supported*.
* **Q5 (disagree on correctness):** Chunk [2] says: *"In case of a verbal complaint, the complaint will be converted to a written complaint by the receiver of the complaint and consent of the complainant will be obtained."* The answer states exactly this, which is the whole expected answer. It should be *correct*. The judge's REASON asks for a point that is not in the expected answer ("informed of their rights"). It also lists "the consent of the complainant will be obtained" as unsupported, although that sentence is in chunk [2].
* **Q8 (disagree):** The generator answered "Not found in the documents", but chunk [1] contains the span: *"e. Retaliation for having reported or threatened to report harassment, or for opposing unlawful harassment, or for participating in an investigation"*. By the judge prompt's own definition, a "Not found" reply to an answerable question is **wrong**. The judge's REASON describes an answer that was never given ("states that retaliation … is included"), so it graded the expected answer instead of the system answer. *Supported = yes* is acceptable, because a "Not found" reply makes no claim. I count the pair as 0 of 2 because the reason it gives for "yes" is the invented answer.

**What the spot-check shows about the 3B judge.** It gave *partly correct* to 9 of the 10 answers and *correct* to none, and its reasons often name points that are not in the expected answer. My reading of all 10 answers gives:
* correct 1 (Q5);
* partly correct 5 (Q1, Q2, Q4, Q9, Q10);
* wrong 4 (Q3, Q6, Q7, Q8);
* supported 8 of 10.

Q1 is not supported: it attributes Bajaj Broking's employee list (chunk [2]) to Niramai. Q6 is not supported: its "broader oversight" claim is not in the chunks. The judge's counts were 0 / 9 / 1 and 9 of 10 supported. A larger judge (Llama-3.1-8B, selectable with `JUDGE=` when there is enough disk) would be the first improvement.

### D2.3 Find the cause (1 mark)
For every query, the table below traces where the complete answer span was lost. There are three possible causes:
* **Candidate retrieval:** the span is not in the best method's top 10, so the reranker never saw it.
* **Reranking:** the span is in the top 10, but not in the final 3 chunks.
* **Generation:** the span is in the 3 chunks the generator received, so the generator caused the error.

After the table come the three chunks of every answer the judge marked wrong, partly correct or unsupported.

In [19]:
def stage(q, b10, a3):
    r10, r3 = retrieval.first_hit_rank(b10, q, chunks), retrieval.first_hit_rank(a3, q, chunks)
    cause = ("candidate retrieval (not in top 10)" if r10 is None else
             "reranking (in top 10, not in final 3)" if r3 is None else "chunks OK - any error is generation")
    return r10, r3, cause
trace = pd.DataFrame([{"qid": q["qid"], f"rank in {BEST} top 10": r10, "rank in final 3": r3, "where the span was lost": cause,
                       "judge": f"{v.correct} / supported={v.supported}"}
                      for q, b, a, (_, v) in zip(queries, before10, after3, verdicts.iterrows())
                      for r10, r3, cause in [stage(q, b, a)]])
trace.to_csv(config.RESULTS / "d2_failure_trace.csv", index=False)
display(trace)
flagged = verdicts[(verdicts.correct != "correct") | (verdicts.supported != "yes")]
for _, v in flagged.iterrows():
    i = next(k for k, q in enumerate(queries) if q["qid"] == v.qid)
    display(Markdown(f"**{v.qid} — {queries[i]['query']}**  \nJudge: {v.correct}, supported = {v.supported}  \n"
                     f"Answer: {answers[i]}  \nExpected: {queries[i]['expected_answer']}  \n"
                     f"Where the span was lost: **{trace.loc[i, 'where the span was lost']}**"))
    for n, c in enumerate(contexts[i], 1):
        print(f"[{n}] ({c.source}) {' '.join(c.text.split())}\n")
if flagged.empty:
    print("No answer was marked wrong, partly correct or unsupported.")

,qid,rank in BM25 top 10,rank in final 3,where the span was lost,judge
0,Q1,1.0,1.0,chunks OK - any error is generation,partly correct / supported=yes
1,Q2,1.0,1.0,chunks OK - any error is generation,partly correct / supported=yes
2,Q3,1.0,2.0,chunks OK - any error is generation,partly correct / supported=yes
3,Q4,1.0,1.0,chunks OK - any error is generation,partly correct / supported=yes
4,Q5,2.0,2.0,chunks OK - any error is generation,partly correct / supported=yes
5,Q6,NaN,NaN,candidate retrieval (not in top 10),partly correct / supported=yes
6,Q7,1.0,NaN,"reranking (in top 10, not in final 3)",wrong / supported=no
7,Q8,1.0,1.0,chunks OK - any error is generation,partly correct / supported=yes
8,Q9,2.0,1.0,chunks OK - any error is generation,partly correct / supported=yes
9,Q10,1.0,1.0,chunks OK - any error is generation,partly correct / supported=yes


**Q1 — Who is covered by Niramai's Code of Conduct?**  
Judge: partly correct, supported = yes  
Answer: Niramai's Code of Conduct is applicable to all employees appointed on the payroll of Niramai Health Analytix Private Limited, including staff, full-time employees both permanent and on probation, temporary employees, trainees, management trainees, apprentices, interns, and third party employees. [1]  
Expected: The code applies to employees appointed on Niramai Health Analytix's payroll. It guides their behavior and attitudes during employment and provides ethical guidance for carrying out official responsibilities, rather than describing a separate policy for outside stakeholders.  
Where the span was lost: **chunks OK - any error is generation**

[1] (niramai_code_of_conduct.txt) Scanned by CamScanner 1. Preamble The Code of Conduct (hereinafter called and referred to as “the Code”) is applicable to all the employees appointed on the payroll of Niramai Health Analytix Private Limited (hereinafter referred to as the “Company”). The Code of Conduct is a comprehensive document that serves the ethical way for the Company’s employees and provides the guidelines by which the employees shall conduct their official duties. The Code has been formulated with the intention of guiding and regulating the conduct and attitude of all the employees of Niramai Health Analytix Pvt. Ltd. during the course of their employment. 2. Objective The object of the Code is to establish appropriate behavioral and conductive norms for the all the employees of Niramai Health Analytix Pvt. Ltd. with the primary focus on inculcating discipline, self-control, integrity, professionalism and Corporate values. The Code aims at creating and building employee’s core

**Q2 — Why did Astral establish a vigil mechanism and whistle-blower policy?**  
Judge: partly correct, supported = yes  
Answer: Astral Limited established a vigil mechanism and whistle-blower policy to encourage employees and directors to report unethical behaviors, malpractices, wrongful conduct, fraud, violation of the company's code of conduct or ethics policy, and violations of law by any employee/director of ASTRAL without fear of retaliation. [1]  
Expected: Astral describes the mechanism as a means for directors and employees to raise concerns about unethical conduct, suspected or actual fraud, and code violations. It also supports corporate governance by helping detect and deter fraudulent activity.  
Where the span was lost: **chunks OK - any error is generation**

[1] (astral_whistle_blower_policy.txt) Page | 1 Vigil Mechanism & Whistle Blower Policy Page | 2 PREAMBLE: ❖ Astral Limited (“ASTRAL” or “Company”) believes in the conduct of the affairs of its business and operations in a fair and transparent manner by adopting highest standards of professionalism, honesty, integrity and ethical behavior. ❖ The Companies Act, 2013 read with rules made thereunder and regulation 22 of SEBI (Listing Obligations and Disclosure requirements) regulations, 2015, mandates that every Listed Company to establish a mechanism for employees/directors to report concerns about unethical behavior, actual or suspected fraud or violation of the company’s code of conduct or ethics policy. ❖ Such vigil mechanism not only helps to detect fraud in organizations, but is also used as a corporate governance tool, which prevents and deters fraudulent activity. ❖ In view of the above, Astral Limited, being a listed company proposes to establish a Vigil Mechanism and to formulat

...

[1] (jiostar_posh_policy.txt) the outcome of the complaint made in good faith, the Complainant and any person providing information or any witness, will be protected from any form of retaliation. Any unwarranted pressures, retaliatory or any other type of unethical behaviour from the Respondent or any other person against the Complainant or witnesses in connection with the complaint or the investigation should be reported by the Complainant to the IC as soon as possible. Disciplinary action will be taken by the Company in relation to any such complaints which are found genuine. Initiate conciliatory process or formal Investigation Upon receipt of the complaint, a copy of the same will be provided within 7 days to the Alleged perpetrator & 10 days time will be given to respond) Management L. Annexure 1. Scope Details a. Verbal conduct such as epithets, derogatory jokes or comments, slurs or unwanted sexual advances, invitations or comments, questions about a person’s sexual practices, o

**Q9 — What initial action does Aurobindo describe after receiving a human-rights complaint?**  
Judge: partly correct, supported = yes  
Answer: After receiving a human-rights complaint, Aurobindo describes that preliminary enquiry will be conducted by involving the complainant in a way that the matter will be dealt with utmost confidentiality within 3 days time. [1]  
Expected: Complaints can arrive through humanrights@aurobindo.com or the Corporate HR hotline numbered 77666. A confidential preliminary inquiry involving the complainant is to occur within three days to establish whether there is an initial case for further examination.  
Where the span was lost: **chunks OK - any error is generation**

[1] (aurobindo_human_rights_policy.txt) . 5.3 Grievance Redressal Mechanism ➢ Any complaint that is received through email address humanrights@aurobindo.com or Hotline Number 77666- Corporate HR would be addressed in a systematic manner. ➢ Preliminary enquiry will be conducted by involving the complainant in such a way that the matter will be dealt with utmost confidentiality within 3days time. Preliminary enquiry will enable the Management to find out whether there is a prima facie case or not. ➢ Corporate HR, Unit HR, concerned Department HOD hold detailed enquiry by having aggrieved employee, contractors, suppliers, vendors from whom the complaint is received. ➢ Corporate/ Unit HR will hold multipartite discussion with the intention of concluding the matter that is being brought to the attention of Management amicably. ➢ In case any repeated complaint with same Employee, Contractor, Vendor matter shall bring an attention to the Director level for appropriate decision . Corporate HR 

**Q10 — How does Aurobindo distinguish the preliminary inquiry from its detailed grievance inquiry?**  
Judge: partly correct, supported = yes  
Answer: The preliminary enquiry involves a preliminary investigation conducted by Corporate HR and involved parties, such as the complainant, to determine if there is a prima facie case. In contrast, the detailed enquiry is a more comprehensive investigation involving Corporate HR, Unit HR, and the concerned Department HOD, where a detailed discussion is held with the aggrieved employee, contractors, suppliers, and vendors. [1] [2]  
Expected: The preliminary stage confidentially involves the complainant and checks whether an initial case exists within three days. The detailed inquiry brings together Corporate HR, Unit HR and the concerned department head with the affected party, followed by discussions aimed at an amicable resolution.  
Where the span was lost: **chunks OK - any error is generation**

[1] (aurobindo_human_rights_policy.txt) . 5.3 Grievance Redressal Mechanism ➢ Any complaint that is received through email address humanrights@aurobindo.com or Hotline Number 77666- Corporate HR would be addressed in a systematic manner. ➢ Preliminary enquiry will be conducted by involving the complainant in such a way that the matter will be dealt with utmost confidentiality within 3days time. Preliminary enquiry will enable the Management to find out whether there is a prima facie case or not. ➢ Corporate HR, Unit HR, concerned Department HOD hold detailed enquiry by having aggrieved employee, contractors, suppliers, vendors from whom the complaint is received. ➢ Corporate/ Unit HR will hold multipartite discussion with the intention of concluding the matter that is being brought to the attention of Management amicably. ➢ In case any repeated complaint with same Employee, Contractor, Vendor matter shall bring an attention to the Director level for appropriate decision . Corporate HR 

**Chosen failure: Q7 (judged wrong). The cause is reranking.** The trace shows the span chunk 652 at **rank 1 in BM25's top 10**, but it is **not in the final 3** after reranking. The cross-encoder ranked JioStar chunk 646, a Bajaj Broking chunk (59) and JioStar chunk 645 above it.

The three chunks the generator received are printed above:
* [1] and [3] are JioStar's definitions, coverage and complaint sections;
* [2] is Bajaj Broking's list of verbal conduct.

None contains JioStar's own list *"verbal… derogatory jokes… visual… offensive pictures, posters, drawings, gestures"*. The generator therefore correctly replied "Not found in the documents". **The needed information was not in the chunks, so this is not a generation error:** candidate retrieval found it, and reranking dropped it.

**For contrast, the other two causes:**
* **Q6 is a candidate-retrieval failure.** The managers' and individuals' responsibilities clause is not in BM25's top 10, so the reranker never saw it. It is not in the top 5 of dense or hybrid search either (Part B table).
* **Q8 is a generation failure.** The span chunk 652 is at rank 1 of the final 3, and it contains the exact retaliation clause. Yet the generator replied "Not found". Q3 is also a generation failure: the span chunk was [2], but the generator compared a course-fee policy instead.

### D2.4 Suggested fix (1 mark)
**Suggested fix for Q7: always keep BM25's first result among the 3 chunks sent to the generator, and use the reranker only to choose the other two.** This may help Q7 and similar queries, because BM25's first result was already correct for 7 of 10 queries (hit rate@1 70%). The reranker's job would then be to add context, not to overrule a strong exact-match result.

I checked the retrieval part with the rankings already saved in `c_rerank_comparison.csv` (BM25 top 3 and reranked top 3 for each query). Keeping BM25's first chunk and filling with the reranker's top results gives:

| Final 3 chunks | Hit rate@1 | Hit rate@3 | Q7 span in final 3 | Time per query (measured parts) |
|---|---|---|---|---|
| BM25 top 10 → rerank → top 3 (used above) | 60% | 80% | no | 1.96 + 21.3 ms |
| BM25 #1 + reranker's best 2 others | 70% | 90% | yes (rank 1) | 1.96 + 21.3 ms |
| BM25 top 3, no reranking | 70% | 90% | yes (rank 1) | 1.96 ms |

The time is the same as the current pipeline: it is the measured BM25 and reranking times from Parts B and C, since nothing extra is computed. This is only the retrieval part. The answer quality was not re-measured, and Q8 shows that a correct chunk [1] does not guarantee a correct answer. So the fix **may help**, but it does not guarantee a better answer. It does not fix Q6: that chunk is not in BM25's top 10, so it needs better candidate retrieval (for example, a stronger embedding model).

For the generation failures (Q3, Q8), a separate change would add a prompt sentence: *"Read all three chunks before deciding the answer is not there,"* or use a larger generator.

**A second retrieval fix that was tested and did not help: a larger candidate pool** (retrieval and reranking only, no generation). The table below shows that pooling dense and BM25 candidates gave the same hit rates (60% / 80%) and doubled the time (36.2 ms vs 18.3 ms per query, measured). Q6 and Q7 are still missing from the final 3: a larger pool does not help when the reranker itself ranks the span chunk too low. The same cross-encoder reranks a pool that combines the dense top 10 and the BM25 top 10, and keeps the top 3. This is compared with the pipeline used above (BM25 top 10 → top 3).

In [20]:
reranker = rr.load_reranker()
def pooled(q):
    pool = list(dict.fromkeys([c for c, _ in dense.search(q["query"], 10)] + [c for c, _ in bm25.search(q["query"], 10)]))
    return rr.rerank(reranker, q["query"], [(c, 0.0) for c in pool], chunks, keep=3), len(pool)
pooled(queries[0])                                             # warm-up
pool_res, pool_ms, pool_sizes = [], [], []
for q in queries:
    t0 = time.perf_counter(); res, n = pooled(q); pool_ms.append(1000 * (time.perf_counter() - t0))
    pool_res.append(res); pool_sizes.append(n)
cur_ms = []
for q in queries:
    t0 = time.perf_counter(); rr.rerank(reranker, q["query"], methods[BEST].search(q["query"], 10), chunks, keep=3)
    cur_ms.append(1000 * (time.perf_counter() - t0))
fix_table = pd.DataFrame({"Pipeline": [f"{BEST} top 10 -> rerank -> top 3 (used above)", "dense top 10 + BM25 top 10 -> rerank -> top 3"],
                          "Hit rate@1 (%)": [retrieval.hit_at(after3, queries, chunks, 1), retrieval.hit_at(pool_res, queries, chunks, 1)],
                          "Hit rate@3 (%)": [retrieval.hit_at(after3, queries, chunks, 3), retrieval.hit_at(pool_res, queries, chunks, 3)],
                          "Candidates reranked (avg)": [10, float(np.mean(pool_sizes))],
                          "Retrieval + rerank time per query (ms)": [round(float(np.mean(cur_ms)), 1), round(float(np.mean(pool_ms)), 1)]})
fix_table.to_csv(config.RESULTS / "d2_fix_test.csv", index=False)
display(fix_table)
display(pd.DataFrame({"qid": [q["qid"] for q in queries],
                      "rank in final 3 (used above)": [retrieval.first_hit_rank(a, q, chunks) for a, q in zip(after3, queries)],
                      "rank in final 3 (pooled)": [retrieval.first_hit_rank(p, q, chunks) for p, q in zip(pool_res, queries)]}))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Pipeline,Hit rate@1 (%),Hit rate@3 (%),Candidates reranked (avg),Retrieval + rerank time per query (ms)
0,BM25 top 10 -> rerank -> top 3 (used above),60.0,80.0,10.0,18.3
1,dense top 10 + BM25 top 10 -> rerank -> top 3,60.0,80.0,15.6,36.2


,qid,rank in final 3 (used above),rank in final 3 (pooled)
0,Q1,1.0,1.0
1,Q2,1.0,1.0
2,Q3,2.0,3.0
3,Q4,1.0,1.0
4,Q5,2.0,2.0
5,Q6,NaN,NaN
6,Q7,NaN,NaN
7,Q8,1.0,1.0
8,Q9,1.0,1.0
9,Q10,1.0,1.0


## Export
`nbconvert` exports the saved `.ipynb`, so save the notebook first (Ctrl+S), or run `python tools/export_html.py` after closing it.

In [ ]:
import subprocess
env = {**os.environ, "JUPYTER_PATH": "/opt/conda/share/jupyter"}
subprocess.run([sys.executable, "-m", "nbconvert", "--to", "html", "notebooks/Group4_Assignment2B_RAG.ipynb",
                "--output-dir", "notebooks"], check=False, env=env)